In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("compas-scores-two-years.csv")

features = [
    "age",
    "priors_count",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "c_charge_degree",
]

X = df[features].copy()
y = df["two_year_recid"]

X["c_charge_degree"] = X["c_charge_degree"].map({"F": 1, "M": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.1%}")

Test accuracy: 69.0%


In [4]:
results = pd.DataFrame({
    "race": df.loc[X_test.index, "race"],
    "actual": y_test,
    "predicted": y_pred,
})

no_new_case = results[results["actual"] == 0]

for group in ["African-American", "Caucasian"]:
    group_rows = no_new_case[no_new_case["race"] == group]

    false_positives = (group_rows["predicted"] == 1).sum()
    total = len(group_rows)
    false_positive_rate = false_positives / total

    print(f"{group}: {false_positives}/{total} = {false_positive_rate:.1%}")

African-American: 89/362 = 24.6%
Caucasian: 35/307 = 11.4%


In [ ]:
#import sqlite3
#compas = sqlite3.connect("compas.db")
#cursor = compas.cursor()
#cursor.execute(""" 
#    CREATE TABLE IF NOT EXISTS compas_data (
#        id INTEGER PRIMARY KEY,
#        sex TEXT NOT NULL CHECK (sex IN ('Male', 'Female')),
#        race TEXT NOT NULL CHECK (race IN ('African-American', 'Caucasian', 'Hispanic', 'Native American', 'Other')),
#        age INTEGER NOT NULL CHECK (age >= 0)
#    ) STRICT;
#""")
#compas.close()